# PWM scanning: how many binding sites does a motif predict?

In the motif analysis notebook we started from **ChIP-seq peaks** and asked which motif is enriched. Here we turn the question around: we start from a **motif (PWM)** and scan a whole genome with it, *without* any peaks. How many sites do we find, and how many of them are real?

The answer is captured by what Wasserman & Sandelin (2004) called the **futility theorem**: scanning a genome with a PWM finds (almost) every real site, but buries it under a huge excess of matches that are never bound. Aerts et al. (2007) put it in numbers: genome-wide PWM scanning "results in a 1000-fold excess of false predictions".

In this notebook we will

1. **predict** the number of false hits for four TFs *before* scanning, using information theory;
2. scan the **yeast genome with GAL4** and look at one true site among thousands of false ones;
3. scan the **human genome with TP53** and ask a hard question: *how do we know a hit is false?* (Verfaillie et al. 2016);
4. look for **other features** that separate bound from unbound sites (GC content, neighbouring weak sites, ...);
5. see how genome composition (**CpG depletion**) changes the count for an **E-box (bHLH)** motif;
6. use **shuffle controls** to show what "by chance" really looks like;
7. collect everything in one **summary table**.

**Kernel:** Python 3. Cells starting with `%%bash` run shell commands, like in the previous notebooks.

**Runtime:** everything runs in a few minutes, except the genome-wide TP53 scan (section 4, ~10 min, cached after the first run).

## 0. Setup

Edit the paths below if your files live elsewhere. The yeast genome and annotation are downloaded from UCSC (small). For human we use the hg19 genome that is already on the server, and the TP53 ChIP-seq peaks from the previous notebooks (MCF7 + Nutlin-3a).

In [ ]:
import os, sys, re, gzip, time, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

USER = os.environ.get("USER", "student")
WORKDIR = f"/data/user/{USER}/pwm_scanning" if os.path.isdir("/data/user") else os.path.expanduser("~/pwm_scanning")

# --- human data already on the server (from the previous notebooks)
HG19_FA   = "/data/teachers/data/genomes/hg19/hg19.fa"
P53_PEAKS = "/data/teachers/data/chip_seq/p53/p53-full_peaks.bed"      # MCF7 + Nutlin-3a TP53 ChIP-seq (hg19)

# --- downloaded on first use
URLS = {
    "sacCer3.fa.gz":       "https://hgdownload.soe.ucsc.edu/goldenPath/sacCer3/bigZips/sacCer3.fa.gz",
    "sgdGene.txt.gz":      "https://hgdownload.soe.ucsc.edu/goldenPath/sacCer3/database/sgdGene.txt.gz",
    "sgdToName.txt.gz":    "https://hgdownload.soe.ucsc.edu/goldenPath/sacCer3/database/sgdToName.txt.gz",
    "chr22.fa.gz":         "https://hgdownload.soe.ucsc.edu/goldenPath/hg19/chromosomes/chr22.fa.gz",   # only if HG19_FA is missing
    # All TP53 ChIP-seq peaks in ChIP-Atlas (hg19, q < 1e-5), one line per peak per experiment
    "TP53_chipatlas.bed":  "https://chip-atlas.dbcls.jp/data/hg19/assembled/Oth.ALL.05.TP53.AllCell.bed",
}

# chromosomes used for the genome-wide TP53 scan (fewer = faster)
TP53_CHROMS = [f"chr{i}" for i in range(1, 23)] + ["chrX"]

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print("working directory:", os.getcwd())

def download(name):
    # Download URLS[name] into the working directory unless it is already there.
    if os.path.exists(name) and os.path.getsize(name) > 0:
        return name
    print(f"downloading {name} ...", flush=True)
    try:
        urllib.request.urlretrieve(URLS[name], name + ".part")
        os.rename(name + ".part", name)
        return name
    except Exception as e:
        print(f"  could not download {name}: {e}")
        return None

The scanning code lives in a small module, `pwmscan.py`. The next cell writes it to disk. You don't have to read it all, but it is short and plain numpy, so nothing is hidden. The key ideas:

* a PWM is converted to **log-odds scores** `log2(p(base | motif) / p(base | background))`;
* the **exact score distribution under the background** is computed by convolution, so every score gets an exact **p-value** (the same idea as FIMO);
* scanning adds one matrix column per position, on **both strands**.

In [ ]:
%%writefile pwmscan.py
"""pwmscan.py - a small, readable PWM scanner for teaching.

Everything here is plain numpy, so you can read exactly what happens:
  * sequences are encoded as integers A=0 C=1 G=2 T=3 N=4
  * a PWM is turned into an integer log-odds matrix (log2 ratios x SCALE)
  * the exact null distribution of scores under a background model gives p-values
    (the same idea FIMO uses)
  * scanning = adding up one matrix column per position, on both strands
"""
import gzip, os, re
import numpy as np
import pandas as pd

BASES = "ACGT"
SCALE = 100            # log2 units x 100, rounded to integers
N_PENALTY = -10**6     # windows containing N can never score

# ---------------------------------------------------------------- sequences
_LUT = np.full(256, 4, dtype=np.uint8)
for i, b in enumerate(BASES):
    _LUT[ord(b)] = i
    _LUT[ord(b.lower())] = i


def encode(seq_bytes):
    """bytes -> uint8 codes (A0 C1 G2 T3 N4) and a soft-mask (lowercase = repeat) array."""
    a = np.frombuffer(seq_bytes, dtype=np.uint8)
    codes = _LUT[a]
    softmask = (a >= ord("a"))          # lowercase letters in UCSC FASTA = RepeatMasker
    return codes, softmask


def read_fasta(path):
    """Read a small FASTA (plain or .gz) into {name: bytes}. Fine for yeast; use FastaIndex for hg19."""
    op = gzip.open if path.endswith(".gz") else open
    seqs, name, buf = {}, None, []
    with op(path, "rb") as fh:
        for line in fh:
            if line.startswith(b">"):
                if name is not None:
                    seqs[name] = b"".join(buf)
                name, buf = line[1:].split()[0].decode(), []
            else:
                buf.append(line.rstrip())
    if name is not None:
        seqs[name] = b"".join(buf)
    return seqs


class FastaIndex:
    """Random access to a large uncompressed FASTA using (or building) a samtools-style .fai."""

    def __init__(self, path):
        self.path = path
        fai = path + ".fai"
        if not os.path.exists(fai):
            fai = self._build_fai(path)
        self.idx = {}
        with open(fai) as fh:
            for line in fh:
                name, length, offset, lb, lw = line.split("\t")[:5]
                self.idx[name] = (int(length), int(offset), int(lb), int(lw))

    @staticmethod
    def _build_fai(path):
        out = os.path.basename(path) + ".fai"          # written to the working directory
        rows, name, length, offset, lb, lw = [], None, 0, 0, 0, 0
        with open(path, "rb") as fh:
            pos = 0
            for line in fh:
                if line.startswith(b">"):
                    if name:
                        rows.append((name, length, offset, lb, lw))
                    name, length, lb, lw = line[1:].split()[0].decode(), 0, 0, 0
                    offset = pos + len(line)
                else:
                    if lb == 0:
                        lb, lw = len(line.rstrip()), len(line)
                    length += len(line.rstrip())
                pos += len(line)
            if name:
                rows.append((name, length, offset, lb, lw))
        with open(out, "w") as fo:
            for r in rows:
                fo.write("\t".join(map(str, r)) + "\n")
        return out

    def chroms(self):
        return list(self.idx)

    def length(self, chrom):
        return self.idx[chrom][0]

    def fetch(self, chrom, start=0, end=None):
        length, offset, lb, lw = self.idx[chrom]
        end = length if end is None else min(end, length)
        start = max(0, start)
        b0 = offset + (start // lb) * lw + start % lb
        b1 = offset + (end // lb) * lw + end % lb
        with open(self.path, "rb") as fh:
            fh.seek(b0)
            raw = fh.read(b1 - b0)
        return raw.replace(b"\n", b"").replace(b"\r", b"")


def base_composition(codes):
    c = np.bincount(codes, minlength=5)[:4].astype(float)
    return c / c.sum()


def dinuc_counts(codes):
    a, b = codes[:-1], codes[1:]
    ok = (a < 4) & (b < 4)
    return np.bincount(a[ok] * 4 + b[ok], minlength=16).reshape(4, 4).astype(float)


# ---------------------------------------------------------------- PWMs
class PWM:
    def __init__(self, name, counts, background=(0.25, 0.25, 0.25, 0.25), pseudocount=0.25):
        """counts: array (width x 4) in A,C,G,T order."""
        self.name = name
        self.counts = np.asarray(counts, dtype=float)
        self.pseudocount = pseudocount
        self.set_background(background)

    def set_background(self, background):
        self.bg = np.asarray(background, dtype=float) / np.sum(background)
        C = self.counts
        self.prob = (C + self.pseudocount) / (C.sum(1, keepdims=True) + 4 * self.pseudocount)
        self.logodds = np.log2(self.prob / self.bg)
        self.int_matrix = np.round(self.logodds * SCALE).astype(np.int32)
        self._null = None
        return self

    @property
    def width(self):
        return self.counts.shape[0]

    @property
    def consensus(self):
        return "".join(BASES[i] for i in self.prob.argmax(1))

    def information_content(self, background=None):
        """Relative entropy (bits) of the motif versus a background - Schneider's R_sequence."""
        bg = self.bg if background is None else np.asarray(background) / np.sum(background)
        return float((self.prob * np.log2(self.prob / bg)).sum())

    def ic_per_position(self):
        return (self.prob * np.log2(self.prob / self.bg)).sum(1)

    # exact null distribution of the integer score under the background
    def null_distribution(self):
        if self._null is None:
            lo = int(self.int_matrix.min(1).sum())
            dist = np.array([1.0])
            offset = 0
            for col in self.int_matrix:
                cmin = int(col.min())
                span = int(col.max() - cmin)
                kernel = np.zeros(span + 1)
                for b in range(4):
                    kernel[col[b] - cmin] += self.bg[b]
                dist = np.convolve(dist, kernel)
                offset += cmin
            tail = np.cumsum(dist[::-1])[::-1]          # P(score >= s)
            self._null = (offset, tail)
        return self._null

    def motif_tail(self):
        """Distribution of scores of sequences drawn from the motif itself: (offset, P(score >= s))."""
        dist, offset = np.array([1.0]), 0
        for col, pr in zip(self.int_matrix, self.prob):
            cmin = int(col.min()); kernel = np.zeros(int(col.max() - cmin) + 1)
            for b in range(4):
                kernel[col[b] - cmin] += pr[b]
            dist = np.convolve(dist, kernel); offset += cmin
        return offset, np.cumsum(dist[::-1])[::-1]

    def threshold_for_sensitivity(self, sensitivity=0.8):
        """Highest score threshold that still recovers `sensitivity` of the sites the motif describes."""
        offset, tail = self.motif_tail()
        ok = np.where(tail >= sensitivity)[0]
        return int(ok[-1] + offset)

    def pvalue(self, int_score):
        offset, tail = self.null_distribution()
        idx = np.clip(np.asarray(int_score) - offset, 0, len(tail) - 1)
        p = tail[idx]
        return np.where(np.asarray(int_score) - offset >= len(tail), 0.0, p)

    def threshold(self, p):
        """Smallest integer score whose tail probability is <= p (None if p is unreachable)."""
        offset, tail = self.null_distribution()
        ok = np.where(tail <= p)[0]
        return None if len(ok) == 0 else int(ok[0] + offset)

    def achieved_p(self, p):
        """The real tail probability at the integer threshold used for nominal p (<= p)."""
        t = self.threshold(p)
        return 0.0 if t is None else float(self.pvalue(t))

    def best_pvalue(self):
        offset, tail = self.null_distribution()
        return float(tail[-1])

    def max_score(self):
        return int(self.int_matrix.max(1).sum())

    def scrambled(self, seed):
        """Column-permuted PWM (Aerts et al. 2007 control): same composition, no real motif."""
        rng = np.random.default_rng(seed)
        perm = rng.permutation(self.width)
        return PWM(f"{self.name}_scr{seed}", self.counts[perm], self.bg, self.pseudocount)


def read_jaspar_counts(text):
    """Parse JASPAR 'A [ 1 2 3 ]' style blocks -> {name: counts(width x 4)}."""
    out, name, rows = {}, None, {}
    for line in text.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if name:
                out[name] = np.array([rows[b] for b in BASES]).T
            name, rows = line[1:].split()[0], {}
        else:
            b = line[0]
            rows[b] = [float(x) for x in re.findall(r"[-\d.]+", line[1:])]
    if name:
        out[name] = np.array([rows[b] for b in BASES]).T
    return out


# ---------------------------------------------------------------- scanning
def _ext(mat):
    """Add a 5th column for N with a huge penalty."""
    return np.hstack([mat, np.full((mat.shape[0], 1), N_PENALTY, dtype=np.int32)]).astype(np.int32)


def score_array(codes, pwm):
    """Best-strand integer score for every window start (length len(codes)-w+1)."""
    w = pwm.width
    n = len(codes) - w + 1
    if n <= 0:
        return np.zeros(0, dtype=np.int32), np.zeros(0, dtype=np.int32)
    fwd = _ext(pwm.int_matrix)
    rev = _ext(pwm.int_matrix[::-1, ::-1])           # reverse complement matrix (ACGT -> TGCA)
    sf = np.zeros(n, dtype=np.int32)
    sr = np.zeros(n, dtype=np.int32)
    for j in range(w):
        window = codes[j:j + n]
        sf += fwd[j][window]
        sr += rev[j][window]
    return sf, sr


def scan(codes, pwm, pval=1e-4, chunk=5_000_000, count_pvals=None):
    """Return start positions, best strand and best-strand integer score of all windows with p <= pval,
    plus the number of strand-specific hits (forward and reverse counted separately, as FIMO does).
    For a palindromic motif one site gives two strand hits but only one position.
    If count_pvals is given, the 4th return value is a dict {p: strand hits at that p}."""
    thr = pwm.threshold(pval)
    if thr is None:
        raise ValueError(f"p={pval} is not reachable for {pwm.name} (best possible p = {pwm.best_pvalue():.2e})")
    w = pwm.width
    extra = {p: pwm.threshold(p) for p in (count_pvals or [])}
    extra_counts = {p: 0 for p in extra}
    pos, strands, scores, n_strand = [], [], [], 0
    for s in range(0, len(codes), chunk):
        sub = codes[s:s + chunk + w - 1]
        if len(sub) < w:
            break
        sf, sr = score_array(sub, pwm)
        n_strand += int((sf >= thr).sum() + (sr >= thr).sum())
        for p, t in extra.items():
            if t is not None:
                extra_counts[p] += int((sf >= t).sum() + (sr >= t).sum())
        sc = np.maximum(sf, sr)
        hit = np.where(sc >= thr)[0]
        pos.append(hit + s)
        strands.append(np.where(sf[hit] >= sr[hit], 1, -1).astype(np.int8))
        scores.append(sc[hit])
    ns = extra_counts if count_pvals else n_strand
    if not pos:
        return np.zeros(0, int), np.zeros(0, np.int8), np.zeros(0, int), ns
    return np.concatenate(pos), np.concatenate(strands), np.concatenate(scores), ns


def scan_genome(seqs, pwm, pval=1e-4, chroms=None, count_pvals=None):
    """seqs: dict chrom -> codes. Returns a DataFrame of hits (one row per position, best strand).
    df.attrs["strand_hits"] holds strand-specific hit counts ({p: n} if count_pvals is given)."""
    rows = []
    n_strand = {p: 0 for p in count_pvals} if count_pvals else 0
    for c in (chroms or seqs.keys()):
        p, st, sc, ns = scan(seqs[c], pwm, pval, count_pvals=count_pvals)
        if count_pvals:
            for k in ns:
                n_strand[k] += ns[k]
        else:
            n_strand += ns
        rows.append(pd.DataFrame({"chrom": c, "start": p, "end": p + pwm.width,
                                  "strand": np.where(st > 0, "+", "-"), "score": sc}))
    df = pd.concat(rows, ignore_index=True)
    df["pval"] = pwm.pvalue(df["score"].values)
    df.attrs["strand_hits"] = n_strand
    return df


def n_scannable(codes, w):
    """Number of windows without N (the 'L' in expected = 2 * L * p)."""
    isn = (codes == 4).astype(np.int32)
    c = np.concatenate([[0], np.cumsum(isn)])
    n = len(codes) - w + 1
    return int(((c[w:w + n] - c[:n]) == 0).sum()) if n > 0 else 0


# ---------------------------------------------------------------- intervals & stats
def merge_intervals(df):
    """Merge overlapping intervals per chromosome; returns {chrom: (starts, ends)} sorted."""
    out = {}
    for c, g in df.groupby("chrom"):
        s = g["start"].values.astype(np.int64)
        e = g["end"].values.astype(np.int64)
        o = np.argsort(s)
        s, e = s[o], e[o]
        ms, me = [s[0]], [e[0]]
        for a, b in zip(s[1:], e[1:]):
            if a <= me[-1]:
                me[-1] = max(me[-1], b)
            else:
                ms.append(a); me.append(b)
        out[c] = (np.array(ms), np.array(me))
    return out


def points_in_intervals(chroms, positions, merged):
    """Boolean: is each (chrom, position) inside any merged interval?"""
    chroms = np.asarray(chroms); positions = np.asarray(positions)
    res = np.zeros(len(positions), dtype=bool)
    for c in np.unique(chroms):
        if c not in merged:
            continue
        m = chroms == c
        s, e = merged[c]
        i = np.searchsorted(s, positions[m], side="right") - 1
        ok = i >= 0
        inside = np.zeros(m.sum(), dtype=bool)
        inside[ok] = positions[m][ok] < e[i[ok]]
        res[m] = inside
    return res


def intervals_with_point(intervals, chroms, positions):
    """Boolean per interval: does it contain at least one of the points?"""
    has = np.zeros(len(intervals), dtype=bool)
    pts = pd.DataFrame({"chrom": chroms, "pos": positions})
    for c, g in intervals.reset_index(drop=True).groupby("chrom"):
        p = np.sort(pts.loc[pts.chrom == c, "pos"].values)
        if len(p) == 0:
            continue
        lo = np.searchsorted(p, g["start"].values, side="left")
        hi = np.searchsorted(p, g["end"].values, side="left")
        has[g.index.values] = hi > lo
    return has


def auc(pos, neg):
    """Area under the ROC curve = P(random positive > random negative) (Mann-Whitney)."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    allv = np.concatenate([pos, neg])
    ranks = pd.Series(allv).rank().values
    rp = ranks[:len(pos)].sum()
    return (rp - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))


def markov1_sequence(codes, length, seed=0):
    """Random sequence with the same dinucleotide (1st-order Markov) statistics as `codes`."""
    D = dinuc_counts(codes)
    T = D / D.sum(1, keepdims=True)
    cum = np.cumsum(T, 1)
    rng = np.random.default_rng(seed)
    u = rng.random(length)
    out = np.empty(length, dtype=np.uint8)
    state = int(rng.choice(4, p=base_composition(codes)))
    cumlist = [list(r) for r in cum]
    for i in range(length):
        r = u[i]; row = cumlist[state]
        state = 0 if r < row[0] else 1 if r < row[1] else 2 if r < row[2] else 3
        out[i] = state
    return out


# ---------------------------------------------------------------- plotting
def draw_logo(ax, pwm, title=None):
    """Classic sequence logo: letter height = probability x information at that position."""
    from matplotlib.textpath import TextPath
    from matplotlib.patches import PathPatch
    from matplotlib.font_manager import FontProperties
    import matplotlib.transforms as mt
    colors = {"A": "#2ca02c", "C": "#1f77b4", "G": "#ff7f0e", "T": "#d62728"}
    fp = FontProperties(family="DejaVu Sans", weight="bold")
    ic = pwm.ic_per_position()
    for i in range(pwm.width):
        heights = pwm.prob[i] * max(ic[i], 0)
        y = 0.0
        for b in np.argsort(heights):
            h = heights[b]
            if h < 0.01:
                continue
            letter = BASES[b]
            tp = TextPath((0, 0), letter, size=1, prop=fp)
            bb = tp.get_extents()
            t = (mt.Affine2D().translate(-bb.x0, -bb.y0)
                 .scale(0.9 / bb.width, h / bb.height).translate(i + 0.05, y))
            ax.add_patch(PathPatch(t.transform_path(tp), color=colors[letter], lw=0))
            y += h
    ax.set_xlim(0, pwm.width)
    ax.set_ylim(0, max(2.2, ic.max() + 0.2))
    ax.set_xticks(np.arange(pwm.width) + 0.5)
    ax.set_xticklabels(range(1, pwm.width + 1), fontsize=7)
    ax.set_ylabel("bits")
    ax.set_title(title or f"{pwm.name}  ({pwm.information_content():.1f} bits)")

In [ ]:
import importlib, pwmscan
importlib.reload(pwmscan)
from pwmscan import *

## 1. Data

### Yeast genome and genes

In [ ]:
download("sacCer3.fa.gz"); download("sgdGene.txt.gz"); download("sgdToName.txt.gz")
yeast_raw = read_fasta("sacCer3.fa.gz")
yeast = {c: encode(s)[0] for c, s in yeast_raw.items() if c != "chrM"}     # skip the mitochondrial genome
yeast_bg = base_composition(np.concatenate(list(yeast.values())))
print(f"{len(yeast)} chromosomes, {sum(len(v) for v in yeast.values())/1e6:.1f} Mb")
print("base composition A,C,G,T:", np.round(yeast_bg, 3), f"-> GC = {yeast_bg[1]+yeast_bg[2]:.1%}")

In [ ]:
%%bash
ls -lh
echo; echo "first lines of the gene table (UCSC sgdGene):"
zcat sgdGene.txt.gz | head -n 3 | cut -f 1-6

In [ ]:
# UCSC sgdGene table: [bin] name chrom strand txStart txEnd ...
g = pd.read_csv("sgdGene.txt.gz", sep="\t", header=None)
chrom_col = next(i for i in range(g.shape[1]) if str(g.iloc[0, i]).startswith("chr"))
g = g.iloc[:, chrom_col - 1:chrom_col + 4]   # name chrom strand txStart txEnd (skips a 'bin' column if present)
g.columns = ["orf", "chrom", "strand", "txStart", "txEnd"]
try:
    names = pd.read_csv("sgdToName.txt.gz", sep="\t", header=None, names=["orf", "gene"])
    g = g.merge(names, on="orf", how="left")
except Exception:
    g["gene"] = np.nan
g["gene"] = g["gene"].fillna(g["orf"])
g["tss"] = np.where(g.strand == "+", g.txStart, g.txEnd)
yeast_genes = g
yeast_genes.head()

### Known GAL4 targets

GAL4 activates the galactose genes through **UAS<sub>GAL</sub>** elements in their promoters. Genome-wide location analysis (Ren et al. 2000) found only about ten direct targets. We use these as our "true" set and define their promoter as 800 bp upstream of the gene start (GAL1 and GAL10 are divergent and share one UAS).

In [ ]:
GAL4_TARGETS = {   # systematic name : gene
    "YBR020W": "GAL1", "YBR019C": "GAL10", "YBR018C": "GAL7", "YLR081W": "GAL2",
    "YML051W": "GAL80", "YDR009W": "GAL3", "YOR120W": "GCY1", "YDR277C": "MTH1",
    "YGL134W": "PCL10", "YBR021W": "FUR4",
}
PROM = 800
def promoters(genes, length=PROM):
    s = np.where(genes.strand == "+", genes.tss - length, genes.tss)
    e = np.where(genes.strand == "+", genes.tss, genes.tss + length)
    return pd.DataFrame({"chrom": genes.chrom.values, "start": np.maximum(s, 0), "end": e,
                         "orf": genes.orf.values, "gene": genes.gene.values})

gal_prom = promoters(yeast_genes[yeast_genes.orf.isin(GAL4_TARGETS)])
gal_prom["gene"] = gal_prom.orf.map(GAL4_TARGETS)
gal_prom

### Human: chr22 (for the fast sections) and the TP53 peaks

In [ ]:
if os.path.exists(HG19_FA):
    hg19 = FastaIndex(HG19_FA)
    chr22_raw = hg19.fetch("chr22")
else:
    print("hg19 not found - downloading chr22 only (the genome-wide TP53 scan will be skipped)")
    hg19 = None
    download("chr22.fa.gz")
    chr22_raw = read_fasta("chr22.fa.gz")["chr22"]
chr22, chr22_rep = encode(chr22_raw)
human_bg = base_composition(chr22)
print(f"chr22: {len(chr22)/1e6:.1f} Mb, of which {(chr22 < 4).sum()/1e6:.1f} Mb is sequence (not N)")
print("base composition A,C,G,T:", np.round(human_bg, 3))

In [ ]:
p53_peaks = pd.read_csv(P53_PEAKS, sep="\t", header=None, comment="#").iloc[:, :3]
p53_peaks.columns = ["chrom", "start", "end"]
p53_peaks = p53_peaks[~p53_peaks.chrom.str.startswith("track")].astype({"start": int, "end": int})
print(len(p53_peaks), "TP53 peaks; median width", int((p53_peaks.end - p53_peaks.start).median()), "bp")
p53_peaks.head()

## 2. The motifs

Count matrices from **JASPAR 2026**:

| motif | JASPAR ID | TF family |
|---|---|---|
| TP53 | MA0106.3 | p53 (tetramer, two half-sites RRRCWWGYYY) |
| GAL4 | MA0299.1 | Zn<sub>2</sub>Cys<sub>6</sub> (dimer, CGG-N<sub>11</sub>-CCG) |
| CTCF | MA0139.2 | C2H2 zinc finger (11 fingers) |
| MAX  | MA0058.4 | bHLH (E-box CACGTG) |

Look at the GAL4 matrix carefully: the GAL4 dimer binds **CGG-N<sub>11</sub>-CCG** (17 bp), but the JASPAR matrix is only 15 columns long and stops after the first C of CCG. We therefore also make an **extended GAL4** matrix by adding the two missing columns (C and G). Matrix quality matters, and we'll see how much.

In [ ]:
JASPAR = '''
>MA0106.3 TP53
A [7544 10514 45 11931 1710 8 244 1228 1145 3851 3584 7104 0 12925 1825 327 879 1472]
C [1037 116 19689 204 374 2 12014 17286 11875 1474 756 26 19350 250 118 33 3338 9183]
G [6563 2433 13 653 137 20393 109 417 2151 9954 17846 8821 4 157 89 20549 116 825]
T [2268 735 837 739 18893 0 7106 2642 3774 2118 1546 190 1 1155 17864 68 17859 9120]
>MA0299.1 GAL4
A [0 0 0 778 292 1019 2339 567 1913 659 96 1170 0 235 0]
C [2779 0 438 655 1137 810 0 1296 335 592 767 517 1437 392 2779]
G [0 2779 2273 927 1151 1002 538 810 0 1744 494 1105 729 419 0]
T [0 0 97 717 498 390 98 194 737 204 1786 404 824 2001 0]
>MA0139.2 CTCF
A [281 56 8 744 40 107 851 5 333 54 12 56 104 372 82]
C [49 800 903 13 528 433 11 0 3 12 0 8 733 13 482]
G [449 21 0 65 334 48 32 903 566 504 890 775 5 507 307]
T [134 36 2 91 11 324 18 3 9 341 8 71 67 17 37]
>MA0058.4 MAX
A [39 6876 0 329 69 0]
C [6876 0 6876 39 159 27]
G [0 87 7 6876 20 6876]
T [0 38 229 19 6876 36]
'''
counts = read_jaspar_counts(JASPAR)
gal4_ext = np.vstack([counts["MA0299.1"], [[0, 2779, 0, 0], [0, 0, 2779, 0]]])   # add the missing C and G

M = {   # each PWM uses the base composition of the genome it will scan as background
    "TP53":     PWM("TP53 (MA0106.3)",      counts["MA0106.3"], human_bg),
    "GAL4":     PWM("GAL4 (MA0299.1)",      counts["MA0299.1"], yeast_bg),
    "GAL4ext":  PWM("GAL4 extended (17 bp)", gal4_ext,          yeast_bg),
    "CTCF":     PWM("CTCF (MA0139.2)",      counts["MA0139.2"], human_bg),
    "MAX":      PWM("MAX (MA0058.4)",       counts["MA0058.4"], human_bg),
}
fig, axes = plt.subplots(len(M), 1, figsize=(8, 1.8 * len(M)))
for ax, (k, pwm) in zip(axes, M.items()):
    draw_logo(ax, pwm)
plt.tight_layout(); plt.show()

## 3. Predict before you scan

How much information does a motif need to pick out its real sites from a genome? Schneider et al. (1986) answered this with two numbers:

* **R<sub>frequency</sub>** = log<sub>2</sub>(number of positions / number of real sites): the information *needed* to find γ sites among G positions (×2 for both strands; palindromes read the same on both strands, so they count once).
* **R<sub>sequence</sub>** = the information content of the motif, measured *against the genome's own base composition* (relative entropy, the sum over the logo).

If R<sub>sequence</sub> < R<sub>frequency</sub>, the motif is not specific enough: on average a random genomic position matches about once every 2<sup>R<sub>sequence</sub></sup> positions, which predicts roughly **2<sup>(R<sub>frequency</sub> − R<sub>sequence</sub>)</sup> matches per real site**.

R<sub>sequence</sub> is the *average* score of a real site, which makes this an idealised, best-case estimate. Real sites vary: to recover most of them you must set the threshold *below* the average site. So we add a **sensitivity-aware** estimate. From the matrix itself we compute the score distribution of sites the motif describes, pick the threshold that still recovers 80% (or 95%) of them, and ask how many chance hits that threshold lets through (2 × G × p).

One more column: a short motif cannot reach every p-value. The **best possible p-value** is the probability of the perfect consensus.

The number of real sites γ is an estimate from the literature; you can change it.

In [ ]:
def palindromic(pwm):
    # Correlation between the motif and its own reverse complement (1 = perfect palindrome)
    a = pwm.prob.ravel(); b = pwm.prob[::-1, ::-1].ravel()
    return np.corrcoef(a, b)[0, 1]

G_YEAST = sum(n_scannable(v, 17) for v in yeast.values())
G_CHR22 = n_scannable(chr22, 18)
G_HG19 = 2.86e9            # non-N hg19 sequence on chr1-22 + X (replaced by the exact value after the scan)

pred = pd.DataFrame([
    # TF       motif      genome   G         gamma (real sites)  source of gamma
    ("GAL4",   "GAL4",    "yeast", G_YEAST,  20,   "~10 target genes x ~2 sites (Ren et al. 2000)"),
    ("GAL4",   "GAL4ext", "yeast", G_YEAST,  20,   "same, extended matrix"),
    ("TP53",   "TP53",    "hg19",  G_HG19,   1148, "strongly bound sites (Verfaillie et al. 2016)"),
    ("CTCF",   "CTCF",    "chr22", G_CHR22,  1500, "rough: ~50k ChIP peaks genome-wide, chr22 ~3%"),
    ("MAX",    "MAX",     "chr22", G_CHR22,  500,  "rough guess, condition dependent"),
], columns=["TF", "motif", "genome", "G", "gamma", "gamma_source"])

rows = []
for _, r in pred.iterrows():
    pwm = M[r.motif]
    pal = palindromic(pwm) > 0.8
    strands = 1 if pal else 2
    r_freq = np.log2(strands * r.G / r.gamma)
    r_seq = pwm.information_content()
    row = {"motif": pwm.name, "genome": r.genome, "width": pwm.width, "palindrome": pal,
           "R_sequence (bits)": round(r_seq, 1), "R_frequency (bits)": round(r_freq, 1),
           "gap (bits)": round(r_freq - r_seq, 1),
           "hits per real site (ideal)": float(f"{2 ** (r_freq - r_seq):.2g}"),
           "best p": f"{pwm.best_pvalue():.1e}"}
    for sens in (0.8, 0.95):
        p = float(pwm.pvalue(pwm.threshold_for_sensitivity(sens)))
        row[f"p for {sens:.0%} of sites"] = f"{p:.1e}"
        row[f"hits per real site ({sens:.0%})"] = float(f"{2 * r.G * p / r.gamma:.2g}")
    rows.append(row)
pred_table = pd.DataFrame(rows)
pred_table.T

**Questions**

1. Which motif is closest to having "enough" information for its genome? Which is furthest away? How much does the answer depend on the sensitivity you want (80% versus 95%)? Compare the TP53 numbers with Verfaillie et al., who predicted ~21,700 sites genome-wide and found ~1,150 strongly bound.
2. The extended GAL4 matrix has only two extra columns. How many bits do they add, and what does that do to the predicted excess?
3. The background matters: GAL4's CGG/CCG is GC-rich, and yeast is AT-rich (38% GC). Does that make GAL4 more or less informative in yeast than in a 50% GC genome? Check with `M["GAL4ext"].information_content([0.25]*4)`.
4. For MAX, even a *perfect* CACGTG is expected to occur by chance thousands of times on one chromosome. What does that tell you about using a 6-bp motif alone?

Write your predictions down; we'll compare them with what we actually observe.

## 4. GAL4 on the whole yeast genome

### 4a. Count the hits

We scan all 16 yeast chromosomes with both GAL4 matrices and keep every window with p < 10<sup>-3</sup>; stricter thresholds are just subsets. We count hits per strand (as FIMO does) so we can compare directly with the chance expectation **2 × L × p**.

Note that with integer scores the p-value jumps in steps, so we use the p-value actually *achieved* at the score threshold.

In [ ]:
PVALS = [1e-3, 1e-4, 1e-5, 1e-6]
gal_hits, rows = {}, []
for k in ["GAL4", "GAL4ext"]:
    t = time.time()
    h = scan_genome(yeast, M[k], pval=1e-3, count_pvals=PVALS)
    h["center"] = h.start + M[k].width // 2
    gal_hits[k] = h
    for p in PVALS:
        expected = 2 * G_YEAST * M[k].achieved_p(p)
        observed = h.attrs["strand_hits"][p]
        sub = h[h.score >= M[k].threshold(p)]
        targets = gal_prom.gene[intervals_with_point(gal_prom, sub.chrom, sub.center)]
        rows.append({"motif": M[k].name, "p": p, "expected by chance": round(expected),
                     "observed (strand hits)": observed, "observed / expected": round(observed / expected, 2),
                     "GAL targets hit": f"{len(targets)}/{len(gal_prom)}"})
    print(f"{M[k].name}: scanned {G_YEAST/1e6:.1f} Mb in {time.time()-t:.1f}s")
gal_table = pd.DataFrame(rows)
gal_table

**Questions**

1. At p < 10<sup>-4</sup>, how does the number of hits compare with what you expect by chance? Did the prediction table get the order of magnitude right?
2. At which threshold do you lose GAL targets? Is there *any* threshold where the true sites outnumber the false ones?
3. Compare the two matrices. Does the extended matrix find more targets, fewer false hits, or both?

### 4b. Sensitivity versus false hits

The plot sums up the futility theorem: as you lower the threshold you recover more true targets, but the number of genome-wide hits grows much faster.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
for k, col in [("GAL4", "tab:gray"), ("GAL4ext", "tab:red")]:
    h, pwm = gal_hits[k], M[k]
    xs, ys = [], []
    for p in np.logspace(-3, -9, 25):
        t = pwm.threshold(p)
        if t is None:
            continue
        sub = h[h.score >= t]
        xs.append(len(sub))
        ys.append(intervals_with_point(gal_prom, sub.chrom, sub.center).sum())
    ax.plot(xs, ys, "o-", color=col, ms=3, label=pwm.name)
ax.set_xscale("log"); ax.set_xlabel("hits in the whole yeast genome"); ax.set_ylabel(f"GAL target promoters hit (of {len(gal_prom)})")
ax.axhline(len(gal_prom), ls=":", color="k"); ax.legend(frameon=False)
ax.set_title("Each point is one p-value threshold"); plt.show()

### 4c. One true locus among many false hits

The GAL1-GAL10 intergenic region contains the best-studied UAS<sub>GAL</sub>, with **four** GAL4 sites. Let's look at 100 kb of chromosome II around it.

In [ ]:
gl1 = yeast_genes[yeast_genes.orf == "YBR020W"].iloc[0]     # GAL1
gl10 = yeast_genes[yeast_genes.orf == "YBR019C"].iloc[0]    # GAL10
mid = (gl1.tss + gl10.tss) // 2
h = gal_hits["GAL4ext"]; pwm = M["GAL4ext"]
near = h[(h.chrom == "chrII") & (h.center > mid - 50_000) & (h.center < mid + 50_000)].copy()
near["mlog10p"] = -np.log10(near.pval)

fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), gridspec_kw={"height_ratios": [1, 1]})
for ax, half in zip(axes, [50_000, 1_000]):
    sub = near[(near.center > mid - half) & (near.center < mid + half)]
    strong = sub.pval < 1e-5
    ax.vlines((sub.center - mid) / 1000, 3, sub.mlog10p, color=np.where(strong, "tab:red", "0.6"), lw=1)
    ax.axvspan((min(gl1.tss, gl10.tss) - mid) / 1000, (max(gl1.tss, gl10.tss) - mid) / 1000, color="gold", alpha=0.3)
    gw = yeast_genes[(yeast_genes.chrom == "chrII") & (yeast_genes.txEnd > mid - half) & (yeast_genes.txStart < mid + half)]
    for _, gg in gw.iterrows():
        x0, x1 = (max(gg.txStart, mid - half) - mid) / 1000, (min(gg.txEnd, mid + half) - mid) / 1000
        ax.plot([x0, x1], [2.6, 2.6], lw=6, color="tab:blue" if gg.strand == "+" else "tab:purple", solid_capstyle="butt")
        if (gg.gene in GAL4_TARGETS.values() and gg.gene not in ("GAL1", "GAL10")) or half < 5000:
            ax.text((x0 + x1) / 2, 2.1, gg.gene, ha="center", va="top", fontsize=8)
    ax.set_xlim(-half / 1000, half / 1000); ax.set_ylim(1.5, max(8, near.mlog10p.max() + 0.5))
    ax.set_ylabel("-log10 p"); ax.set_xlabel("kb from the GAL1-GAL10 intergenic midpoint")
axes[0].set_title(f"GAL4 (extended) hits with p < 1e-3 on chrII: {len(near)} in 100 kb; red = p < 1e-5; yellow = GAL1-GAL10 UAS")
plt.tight_layout(); plt.show()

**Questions**

1. Is the strongest hit in these 100 kb inside the UAS? How many hits elsewhere in the window are just as strong?
2. What is different about the true UAS when you zoom in (bottom panel)?

### 4d. Clusters beat single sites

The real UAS contains several sites close together. We give every 500-bp window a **cluster score**: the sum, over all hits inside it, of how far each hit exceeds the scanning threshold, −log<sub>10</sub>(p) − 3. A window full of barely significant hits scores low, while a few good sites together score high. We compare this with ranking by the **single best hit**. This is the simplest version of what HMM-based tools such as Cluster-Buster do; the next notebook (CRM scanning) is about those.

In [ ]:
def window_scores(hits, window=500):
    # For every hit, score the window [hit, hit+window): cluster = sum of -log10 p, single = best -log10 p
    out = []
    for c, g in hits.groupby("chrom"):
        g = g.sort_values("center")
        pos = g.center.values; w = -np.log10(g.pval.values) - 3     # excess over the p = 1e-3 threshold
        cs = np.concatenate([[0], np.cumsum(w)])
        j = np.searchsorted(pos, pos + window)
        best = np.array([w[i:k].max() for i, k in zip(range(len(pos)), j)]) + 3
        out.append(pd.DataFrame({"chrom": c, "start": pos, "end": pos + window,
                                 "cluster": cs[j] - cs[:len(pos)], "single": best, "n_sites": j - np.arange(len(pos))}))
    return pd.concat(out, ignore_index=True)

def top_windows(ws, col, n=300):
    # Greedy: take the best window, discard overlapping ones, repeat
    picked, taken = [], {}
    for _, r in ws.sort_values(col, ascending=False).iterrows():
        tk = taken.setdefault(r.chrom, [])
        if any(abs(r.start - s) < 500 for s in tk):
            continue
        tk.append(r.start); picked.append(r)
        if len(picked) >= n:
            break
    return pd.DataFrame(picked).reset_index(drop=True)

def nearest_gene(chrom, pos):
    g = yeast_genes[yeast_genes.chrom == chrom]
    i = (g.tss - pos).abs().idxmin()
    return g.loc[i, "gene"]

ws = window_scores(gal_hits["GAL4ext"])
fig, ax = plt.subplots(figsize=(6.5, 4))
for col, color in [("cluster", "tab:red"), ("single", "0.5")]:
    top = top_windows(ws, col)
    found = []
    for i in range(len(top)):
        # a window counts for every target promoter it overlaps
        ov = gal_prom[(gal_prom.chrom == top.chrom[i]) & (gal_prom.start < top.end[i]) & (gal_prom.end > top.start[i])]
        found.append(set(ov.gene))
    cum = [len(set().union(*found[:n + 1])) for n in range(len(found))]
    ax.plot(np.arange(1, len(cum) + 1), cum, color=color, label=f"rank by {col}")
ax.set_xscale("log"); ax.set_xlabel("top N windows in the genome"); ax.set_ylabel("GAL target promoters found")
ax.legend(frameon=False); plt.show()

top = top_windows(ws, "cluster", 15)
top["nearest gene"] = [nearest_gene(c, s + 250) for c, s in zip(top.chrom, top.start)]
top[["chrom", "start", "n_sites", "cluster", "single", "nearest gene"]].round(1)

**Questions**

1. Which GAL targets are in the top 15 windows by cluster score? Which targets are missed, and why might they be (one site only, a weaker site, a site we did not model)?
2. Look at the non-GAL windows in the top 15. Would you call them false positives, or candidate new targets? What experiment would you do to decide?
3. Clustering helps GAL4. Keep this in mind for TP53 below.

## 5. TP53 on the human genome

### 5a. Genome-wide scan

TP53 binds as a tetramer to two half-sites, RRRCWWGYYY RRRCWWGYYY, with no spacer (Verfaillie et al. 2016). Its matrix carries ~21 bits, much more than GAL4. Is that enough for a genome 250 times larger than yeast's?

The scan takes ~10 min the first time; the hits are saved and reloaded afterwards.

In [ ]:
CACHE = "tp53_hg19_hits.tsv.gz"; STATS = "tp53_hg19_stats.json"
TP53_PVALS = [1e-4, 1e-5, 1e-6, 1e-7, 1e-8]
pwm53 = M["TP53"]
import json
if os.path.exists(CACHE) and os.path.exists(STATS):
    tp53 = pd.read_csv(CACHE, sep="\t")
    stats = json.load(open(STATS)); stats["strand_hits"] = {float(k): v for k, v in stats["strand_hits"].items()}
elif hg19 is not None:
    parts, stats = [], {"L": 0, "strand_hits": {p: 0 for p in TP53_PVALS}}
    t0 = time.time()
    for c in TP53_CHROMS:
        codes, rep = encode(hg19.fetch(c))
        p, st, sc, ns = scan(codes, pwm53, 1e-4, count_pvals=TP53_PVALS)
        stats["L"] += n_scannable(codes, pwm53.width)
        for k in ns: stats["strand_hits"][k] += ns[k]
        parts.append(pd.DataFrame({"chrom": c, "start": p, "strand": np.where(st > 0, "+", "-"),
                                   "score": sc, "repeat": rep[p + pwm53.width // 2]}))
        print(f"{c}: {len(p):,} hits   ({time.time()-t0:.0f}s)", flush=True)
        del codes, rep
    tp53 = pd.concat(parts, ignore_index=True)
    tp53.to_csv(CACHE, sep="\t", index=False)
    json.dump({"L": stats["L"], "strand_hits": {str(k): v for k, v in stats["strand_hits"].items()}}, open(STATS, "w"))
else:
    raise SystemExit("hg19 is needed for this section")
tp53["end"] = tp53.start + pwm53.width
tp53["center"] = tp53.start + pwm53.width // 2
tp53["pval"] = pwm53.pvalue(tp53.score.values)
G_HG19 = stats["L"]
print(f"{len(tp53):,} positions with p < 1e-4 in {G_HG19/1e9:.2f} Gb")

In [ ]:
merged_peaks = merge_intervals(p53_peaks)
peak_bp = (p53_peaks.end - p53_peaks.start).sum()
rows = []
for p in TP53_PVALS:
    t = pwm53.threshold(p); sub = tp53[tp53.score >= t]
    inpk = points_in_intervals(sub.chrom.values, sub.center.values, merged_peaks)
    exp = 2 * G_HG19 * pwm53.achieved_p(p)
    rows.append({"p": p, "expected by chance": round(exp), "observed (strand hits)": stats["strand_hits"][p],
                 "obs/exp": round(stats["strand_hits"][p] / exp, 2),
                 "positions": len(sub), "in a TP53 peak": int(inpk.sum()),
                 "precision (% in peaks)": round(100 * inpk.mean(), 2),
                 "recall (% peaks with a hit)": round(100 * intervals_with_point(p53_peaks, sub.chrom, sub.center).mean(), 1),
                 "enrichment in peaks": round(inpk.mean() / (peak_bp / G_HG19), 0)})
tp53_table = pd.DataFrame(rows)
tp53_table

**Questions**

1. At p < 10<sup>-4</sup>, is the genome-wide hit count different from chance at all? At what threshold does the excess over chance become visible?
2. The previous notebook found the TP53 motif in ~64% of the peaks (RSAT). Which threshold here gives a similar recall, and what fraction of the genome-wide hits at that threshold are in a peak?
3. The enrichment in peaks is huge, yet the precision is tiny. How can both be true?

### 5b. How do we know a hit is "false"?

So far "false" meant "not in *our* peaks". But one ChIP-seq experiment is one cell line, one treatment and one antibody. Maybe the site is used in another cell type? We can never prove that a site is *never* used. What we can do is collect evidence, as Verfaillie et al. (2016) did with 15 public TP53 ChIP-seq data sets:

* if a site is bound in (nearly) **every** experiment, across cell types and stimuli, it is a robust TP53 site;
* if a site is bound in **none** of many experiments, it is very unlikely to be a functional TP53 site, at least not in the cell-autonomous DNA-damage response these experiments probe.

They found ~1,150 strongly bound, ~3,150 weakly bound and ~17,400 never-bound sites among ~21,700 predicted sites.

We do the same with **ChIP-Atlas** (Oki et al. 2018), which reprocessed all public TP53 ChIP-seq experiments. Each strong motif match (p < 10<sup>-5</sup>) gets a count: in how many experiments is it inside a peak? The file is large (it can be a few hundred MB). If the download fails, the notebook falls back to the course peaks only, and the classes become much less reliable.

In [ ]:
CAND_P = 1e-5
cand = tp53[tp53.pval <= CAND_P].reset_index(drop=True).copy()
cand["mlog10p"] = -np.log10(cand.pval)

ca_file = download("TP53_chipatlas.bed")
HAVE_CA = False
if ca_file:
    try:
        skip = 1 if open(ca_file).readline().startswith("track") else 0
        ca = pd.read_csv(ca_file, sep="\t", header=None, usecols=[0, 1, 2, 3], dtype=str, skiprows=skip)
        ca = ca[ca[0].isin(TP53_CHROMS)]
        ca.columns = ["chrom", "start", "end", "name"]
        ca[["start", "end"]] = ca[["start", "end"]].astype(int)
        ca["exp"] = ca.name.str.extract(r"ID=([SDE]RX\d+)", expand=False)
        from urllib.parse import unquote
        cellmap = {n: (re.search(r"@\s*([^);]+)", unquote(n)) or [None, "unknown"])[1].strip() for n in ca.name.unique()}
        ca["cell"] = ca.name.map(cellmap)
        ca = ca.dropna(subset=["exp"]).drop(columns="name")
        HAVE_CA = ca.exp.nunique() > 1
    except Exception as e:
        print("could not parse the ChIP-Atlas file:", e)

if HAVE_CA:
    exps = ca.exp.unique()
    cand["n_exp"] = 0; cand["n_cell"] = 0
    for e, g in ca.groupby("exp"):
        cand["n_exp"] += points_in_intervals(cand.chrom.values, cand.center.values, merge_intervals(g))
    for c, g in ca.groupby("cell"):
        cand["n_cell"] += points_in_intervals(cand.chrom.values, cand.center.values, merge_intervals(g))
    N_EXP = len(exps)
    print(f"ChIP-Atlas: {N_EXP} TP53 experiments in {ca.cell.nunique()} cell types, {len(ca):,} peaks")
else:
    print("Using only the course peaks (one experiment) - the classes below are much less reliable.")
    cand["n_exp"] = points_in_intervals(cand.chrom.values, cand.center.values, merged_peaks).astype(int)
    cand["n_cell"] = cand.n_exp
    N_EXP = 1

In [ ]:
%%bash
# what one line of the ChIP-Atlas file looks like (column 4 holds the experiment ID and cell type)
[ -f TP53_chipatlas.bed ] && head -n 3 TP53_chipatlas.bed | cut -c 1-250 || echo "no ChIP-Atlas file"

In [ ]:
STRONG_MIN = max(1, int(round(0.2 * N_EXP)))      # "strongly bound" = in a peak in >= 20% of experiments
cand["class"] = np.select([cand.n_exp >= STRONG_MIN, cand.n_exp > 0], ["strong", "weak"], "unbound")
print(cand["class"].value_counts().reindex(["strong", "weak", "unbound"]).to_string())
print(f"(strong = bound in >= {STRONG_MIN} of {N_EXP} experiments)")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(cand.n_exp, bins=np.arange(N_EXP + 2) - 0.5, color="tab:blue")
axes[0].set_yscale("log"); axes[0].set_xlabel("number of TP53 ChIP-seq experiments with a peak on the site")
axes[0].set_ylabel("motif matches (p < 1e-5)")
bins = pd.qcut(cand.mlog10p, 10, duplicates="drop")
frac = cand.groupby(bins, observed=True).agg(any_peak=("n_exp", lambda x: (x > 0).mean()),
                                             strong=("class", lambda x: (x == "strong").mean()))
xm = [b.mid for b in frac.index]
axes[1].plot(xm, frac.any_peak, "o-", label="bound in >= 1 experiment")
axes[1].plot(xm, frac.strong, "o-", label="strongly bound")
axes[1].set_xlabel("motif strength, -log10 p (deciles)"); axes[1].set_ylabel("fraction of matches")
axes[1].legend(frameon=False); plt.tight_layout(); plt.show()

**Questions**

1. What fraction of strong TP53 motif matches are never bound in any experiment? Compare with Verfaillie et al. (~80% of their predicted sites were unbound).
2. Does binding increase with motif strength? Is even the best decile mostly bound?

### 5c. The reverse check: peaks without a motif

Verfaillie et al. found that ChIP-seq peaks *without* a TP53 motif ("indirect" peaks) are mostly unique to one experiment, while peaks *with* a motif are found again and again. If that's true, the two kinds of error behave differently: motif hits without binding are a sequence problem, and binding without a motif looks more like a ChIP artefact.

In [ ]:
pk = p53_peaks.copy()
hits4 = tp53[tp53.pval <= 1e-4]
pk["direct"] = intervals_with_point(pk, hits4.chrom, hits4.center)
pk["center"] = (pk.start + pk.end) // 2
if HAVE_CA:
    pk["n_exp"] = 0
    for e, g in ca.groupby("exp"):
        pk["n_exp"] += points_in_intervals(pk.chrom.values, pk.center.values, merge_intervals(g))
    fig, ax = plt.subplots(figsize=(6, 3.8))
    for d, col in [(True, "tab:red"), (False, "0.5")]:
        x = np.sort(pk.n_exp[pk.direct == d]); y = 1 - np.arange(len(x)) / len(x)
        ax.step(x, y, where="post", color=col, label=f"{'with' if d else 'without'} motif (n={len(x)})")
    ax.set_xlabel("experiments (of %d) with a peak at the same place" % N_EXP); ax.set_ylabel("fraction of our peaks with at least x")
    ax.legend(frameon=False); plt.show()
    print(pk.groupby("direct").n_exp.median().rename("median number of experiments"))
else:
    print(pk.direct.value_counts().rename({True: "with motif", False: "without motif"}))

### 5d. Honest caveats

"Bound in none of N experiments" is strong evidence, not proof:

* **Mappability.** ChIP-seq reads cannot be placed uniquely in some repeats, so a site there can look unbound even if it is bound. TP53 sites are often found in LTR and Alu repeats (Verfaillie et al.; Cui et al. 2011). The hg19 FASTA marks repeats in lowercase, so we can check how often each class sits in a repeat.
* **Conditions.** All experiments stabilise TP53 in cultured cells. Sites used only in development or *in vivo* would be missed.
* **Binding is not function.** Only ~40% of TP53 ChIP-seq peaks drove a reporter in CHEQ-seq, almost all of them the ones with a motif.

In [ ]:
print(cand.groupby("class").repeat.mean().reindex(["strong", "weak", "unbound"]).rename("fraction in a repeat (lowercase)").round(2))

## 6. What else distinguishes a real site?

If the motif match is the same, why is one site bound in every experiment and another never? Some ideas you may have heard:

* real sites sit in **GC-rich** or **CpG-rich** surroundings;
* real sites are surrounded by **many weak sites** that help recruit the TF (homotypic clustering, as for GAL4 above);
* real sites are in **repeats** less (or more) often.

We test each one on strongly bound versus unbound TP53 matches. One trap: strongly bound sites also have **better motif scores** (5b). Any feature that correlates with motif score would look predictive for that reason alone. So we also compare **score-matched** sets: every strongly bound site is paired with an unbound site of (nearly) the same motif score.

The **AUC** summarises each comparison: 0.5 = no separation, 1.0 = perfect.

In [ ]:
FLANK = 500
rng = np.random.default_rng(0)
def sample(df, n): return df.sample(min(n, len(df)), random_state=1)
strong = sample(cand[cand["class"] == "strong"], 1500)
unbound_all = cand[cand["class"] == "unbound"]
unbound = sample(unbound_all, 1500)

# score-matched unbound set: nearest score, without replacement
pool = unbound_all.sort_values("score").reset_index(drop=True)
used = np.zeros(len(pool), bool); matched_idx = []
for s in strong.score.values:
    i = np.searchsorted(pool.score.values, s)
    for j in sorted(range(max(0, i - 50), min(len(pool), i + 50)), key=lambda j: abs(pool.score.values[j] - s)):
        if not used[j]:
            used[j] = True; matched_idx.append(j); break
unbound_matched = pool.iloc[matched_idx]

thr_weak = pwm53.threshold(1e-3)
def features(sites):
    gc_prof, rows = [], []
    for _, r in sites.iterrows():
        raw = hg19.fetch(r.chrom, r.center - FLANK, r.center + FLANK) if hg19 else chr22_raw[r.center - FLANK:r.center + FLANK]
        codes, rep = encode(raw)
        if len(codes) < 2 * FLANK:
            continue
        isgc = np.where(codes < 4, ((codes == 1) | (codes == 2)).astype(float), np.nan)
        gc_prof.append(isgc)
        n = (codes < 4).sum()
        c, g = (codes == 1).sum(), (codes == 2).sum()
        cg = ((codes[:-1] == 1) & (codes[1:] == 2)).sum()
        sf, sr = score_array(codes, pwm53)
        offs = np.arange(len(sf)) - (FLANK - pwm53.width // 2)
        weak = ((np.maximum(sf, sr) >= thr_weak) & (np.abs(offs) > 20)).sum()
        flank = np.r_[isgc[FLANK - 150:FLANK - 50], isgc[FLANK + 50:FLANK + 150]]
        rows.append({"motif score (-log10 p)": r.mlog10p, "GC flanks 50-150 bp": np.nanmean(flank),
                     "GC +-500 bp": np.nanmean(isgc), "CpG obs/exp +-500 bp": cg * n / max(c * g, 1),
                     "weak TP53 sites (p<1e-3) +-500 bp": weak, "repeat fraction +-500 bp": rep.mean()})
    return pd.DataFrame(rows), np.array(gc_prof)

assert len(strong) >= 20 and len(unbound) >= 20, "too few strongly bound or unbound sites - check section 5b"
t = time.time()
F = {name: features(df) for name, df in [("strong", strong), ("unbound", unbound), ("unbound, score-matched", unbound_matched)]}
print(f"features computed in {time.time()-t:.0f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
k = np.ones(25) / 25
for name, col in [("strong", "tab:red"), ("unbound", "0.5"), ("unbound, score-matched", "tab:blue")]:
    prof = np.nanmean(F[name][1], axis=0)
    ax.plot(np.arange(-FLANK, FLANK)[12:-12], np.convolve(prof, k, mode="valid"), color=col, label=name)
ax.set_xlabel("bp from the TP53 motif"); ax.set_ylabel("GC fraction (25-bp smoothed)"); ax.legend(frameon=False)
plt.show()

feat_cols = list(F["strong"][0].columns)
fig, axes = plt.subplots(1, len(feat_cols), figsize=(3 * len(feat_cols), 3.2))
for ax, f in zip(axes, feat_cols):
    data = [F[n][0][f].dropna() for n in ["strong", "unbound", "unbound, score-matched"]]
    ax.boxplot(data, showfliers=False); ax.set_xticks([1, 2, 3]); ax.set_xticklabels(["strong", "unb.", "unb.\nmatched"])
    ax.set_title(f, fontsize=9)
plt.tight_layout(); plt.show()

auc_table = pd.DataFrame({f: {"strong vs unbound": auc(F["strong"][0][f], F["unbound"][0][f]),
                              "strong vs score-matched unbound": auc(F["strong"][0][f], F["unbound, score-matched"][0][f])}
                          for f in feat_cols}).T.round(2)
auc_table

**Questions**

1. Which features separate strong from unbound sites at all (AUC clearly above or below 0.5)? Which feature is the best?
2. Which of those survive score matching? What does it mean when a feature's AUC drops to ~0.5 after matching?
3. Do strongly bound TP53 sites have more weak TP53 sites around them? Compare with what clustering did for GAL4 in section 4d.
4. Verfaillie et al. found that GC content, dinucleotides and DNA shape differ *on average* around bound sites, but add almost nothing to the motif for prediction (motif alone AUC 0.87; motif + everything 0.89). Does your table agree?

TP53 turns out to be an "unsophisticated" factor: one good site is enough, with no clusters and no partner motifs needed. Developmental enhancers work differently, and that's the topic of the CRM-scanning notebook.

## 7. An E-box (bHLH) motif and the missing CpGs

bHLH factors such as MAX, MYC and USF bind the **E-box CACGTG**: only 6 bp, ~12 bits. From the prediction table, even a perfect match is expected thousands of times per chromosome. But E-boxes come in variants, and **CACGTG contains a CpG**.

In vertebrate genomes CpG dinucleotides are depleted: methylated C in CpG mutates to T, so CpGs are lost over evolution except in CpG islands. A model that only knows base frequencies will therefore *over*-predict CACGTG. We compare it with **CAGCTG**, another palindromic E-box with exactly the same base composition but no CpG.

In [ ]:
def kmer_count(codes, kmer):
    k = len(kmer); n = len(codes) - k + 1
    ok = np.ones(n, bool)
    for j, b in enumerate(kmer):
        ok &= codes[j:j + n] == "ACGT".index(b)
    return int(ok.sum())

mono = base_composition(chr22)
D = dinuc_counts(chr22); T = D / D.sum(1, keepdims=True)
L22 = n_scannable(chr22, 6)
def expected(kmer, model):
    idx = ["ACGT".index(b) for b in kmer]
    if model == "mono":
        return L22 * np.prod(mono[idx])
    return L22 * mono[idx[0]] * np.prod([T[a, b] for a, b in zip(idx[:-1], idx[1:])])

cpg_oe = D[1, 2] / D.sum() / (mono[1] * mono[2])
print(f"CpG observed/expected on chr22: {cpg_oe:.2f}")
ebox = pd.DataFrame([{"k-mer": km, "CpG": "CG" in km, "observed": kmer_count(chr22, km),
                      "expected (base composition)": round(expected(km, "mono")),
                      "expected (dinucleotides)": round(expected(km, "di"))} for km in ["CACGTG", "CAGCTG"]])
ebox["obs / exp (bases)"] = (ebox.observed / ebox["expected (base composition)"]).round(2)
ebox["obs / exp (dinucs)"] = (ebox.observed / ebox["expected (dinucleotides)"]).round(2)
ebox

Where are the CACGTG E-boxes that *did* survive? CpG islands are unmethylated, so CpGs persist there, and bHLH factors such as MYC/MAX bind unmethylated E-boxes much better than methylated ones. We mark CpG-island-like 500-bp windows (GC > 50% and CpG obs/exp > 0.6, as in Gardiner-Garden & Frommer 1987) and ask what fraction of each E-box falls in them.

In [ ]:
W = 500
nb = len(chr22) // W
blk = chr22[:nb * W].reshape(nb, W)
nN = (blk == 4).sum(1); c = (blk == 1).sum(1); g = (blk == 2).sum(1)
cg = ((blk[:, :-1] == 1) & (blk[:, 1:] == 2)).sum(1)
gc = (c + g) / W
oe = np.where(c * g > 0, cg * W / np.maximum(c * g, 1), 0)
island = (nN == 0) & (gc > 0.5) & (oe > 0.6)
print(f"CpG-island-like windows cover {island.sum() / (nN == 0).sum():.1%} of chr22")

def frac_in_islands(kmer):
    k = len(kmer); n = len(chr22) - k + 1
    ok = np.ones(n, bool)
    for j, b in enumerate(kmer):
        ok &= chr22[j:j + n] == "ACGT".index(b)
    pos = np.where(ok)[0]; pos = pos[pos // W < nb]
    return island[pos // W].mean()
for km in ["CACGTG", "CAGCTG"]:
    print(f"{km}: {frac_in_islands(km):.1%} of occurrences are in CpG-island-like windows")

pwmmax = M["MAX"]
print(f"\nMAX matrix: best possible p = {pwmmax.best_pvalue():.1e}")
try:
    scan(chr22, pwmmax, 1e-4)
except ValueError as e:
    print("p < 1e-4:", e)
p_, s_, sc_, n_ = scan(chr22, pwmmax, 1e-3)
print(f"p < 1e-3: {n_:,} strand hits on chr22, expected by chance {2 * L22 * pwmmax.achieved_p(1e-3):,.0f}")

**Questions**

1. Which model predicts the CACGTG count well, and which does not? What about CAGCTG?
2. Even after CpG depletion, how many CACGTG sites remain on this one chromosome? MAX/MYC ChIP-seq typically finds a few thousand peaks *genome-wide*. Which E-boxes would you expect to be bound?
3. Why is a PWM scan with a 6-bp motif essentially a k-mer count?

## 8. Shuffle controls: what does "by chance" look like?

So far "chance" has been a formula. Two experimental controls make it concrete:

* **Scrambled PWMs** (Aerts et al. 2007): permute the columns of the matrix. Base composition and information content stay identical, but it is no longer a real motif. If the real PWM finds no more hits than its scrambled versions, the hits are not about the motif.
* **Dinucleotide-matched random sequence**: a random chr22-like sequence with the same dinucleotide frequencies (a first-order Markov chain). It keeps composition and CpG depletion, but loses everything else, including repeats and real sites.

In [ ]:
N_SCR = 5
rows = []
for key, genome, codes_list, p in [("GAL4ext", "yeast", list(yeast.values()), 1e-4),
                                    ("TP53", "chr22", [chr22], 1e-4),
                                    ("MAX", "chr22", [chr22], 1e-3)]:
    pwm = M[key]
    real = sum(scan(cd, pwm, p)[3] for cd in codes_list)
    scr = [sum(scan(cd, pwm.scrambled(s), p)[3] for cd in codes_list) for s in range(N_SCR)]
    L = sum(n_scannable(cd, pwm.width) for cd in codes_list)
    rows.append({"motif": pwm.name, "genome": genome, "p": p, "expected": round(2 * L * pwm.achieved_p(p)),
                 "real PWM": real, "scrambled PWMs (mean)": round(np.mean(scr)), "scrambled (min-max)": f"{min(scr)}-{max(scr)}"})
scr_table = pd.DataFrame(rows)
scr_table

In [ ]:
RAND_LEN = 10_000_000
t = time.time()
rand22 = markov1_sequence(chr22[chr22 < 4], RAND_LEN, seed=1)
print(f"generated {RAND_LEN/1e6:.0f} Mb of dinucleotide-matched sequence in {time.time()-t:.0f}s")
L_real = n_scannable(chr22, 18) / 1e6
rows = []
for key, p in [("TP53", 1e-4), ("TP53", 1e-6), ("MAX", 1e-3)]:
    pwm = M[key]
    real = scan(chr22, pwm, p)[3] / (n_scannable(chr22, pwm.width) / 1e6)
    rnd = scan(rand22, pwm, p)[3] / (RAND_LEN / 1e6)
    rows.append({"motif": pwm.name, "p": p, "expected per Mb": round(2e6 * pwm.achieved_p(p), 2),
                 "real chr22 per Mb": round(real, 2), "random (dinucleotide) per Mb": round(rnd, 2)})
pd.DataFrame(rows)

Now compare the genome with the TP53 **peaks**: real versus scrambled TP53 matrices, hits per Mb.

In [ ]:
if hg19 is not None:
    peak_seq = np.concatenate([np.append(encode(hg19.fetch(r.chrom, r.start, r.end))[0], 4)
                               for r in p53_peaks.itertuples()])
    Lp = n_scannable(peak_seq, 18) / 1e6
    rows = []
    for name, pwm in [("TP53 (real)", pwm53)] + [(f"TP53 scrambled {s}", pwm53.scrambled(s)) for s in range(N_SCR)]:
        in_peaks = scan(peak_seq, pwm, 1e-4)[3] / Lp
        in_chr22 = scan(chr22, pwm, 1e-4)[3] / L_real
        rows.append({"matrix": name, "hits per Mb in peaks": round(in_peaks, 1), "hits per Mb on chr22": round(in_chr22, 1),
                     "enrichment in peaks": round(in_peaks / in_chr22, 1)})
    peak_ctrl = pd.DataFrame(rows)
    display(peak_ctrl)

**Questions**

1. Genome-wide, does the real TP53 matrix find more hits than its scrambled versions? And inside the peaks?
2. Real chr22 and the random sequence have the same dinucleotide composition. Where do any differences in hit density come from?
3. Put it in one sentence: when is a PWM hit "more than chance"?

## 9. Summary

In [ ]:
ctcf_hits = scan(chr22, M["CTCF"], 1e-4)[3]
def row(tf, key, genome, observed, expected, scrambled, recovered, rescue):
    pr = pred_table.set_index("motif").loc[M[key].name]
    return {"TF": tf, "genome": genome, "motif bits": pr["R_sequence (bits)"], "bits needed": pr["R_frequency (bits)"],
            "hits per real site (80%)": pr["hits per real site (80%)"], "observed hits": int(observed),
            "expected by chance": int(expected), "scrambled PWM": scrambled, "true sites recovered": recovered,
            "what rescues prediction?": rescue}

st = scr_table.set_index("motif")
g4 = gal_table[(gal_table.p == 1e-4) & (gal_table.motif == M["GAL4ext"].name)].iloc[0]
t4 = tp53_table[tp53_table.p == 1e-4].iloc[0]
summary = pd.DataFrame([
    row("GAL4", "GAL4ext", "yeast (p<1e-4)", g4["observed (strand hits)"], g4["expected by chance"],
        st.loc[M["GAL4ext"].name, "scrambled PWMs (mean)"], f"{g4['GAL targets hit']} target promoters",
        "clusters of sites (UAS)"),
    row("TP53", "TP53", "hg19 (p<1e-4)", t4["observed (strand hits)"], t4["expected by chance"],
        "see section 8", f"{t4['recall (% peaks with a hit)']}% of peaks",
        "a single strong site; reproducible binding across conditions"),
    row("CTCF", "CTCF", "chr22 (p<1e-4)", ctcf_hits, round(2 * G_CHR22 * M["CTCF"].achieved_p(1e-4)), "-", "not tested",
        "high information content (and many real sites)"),
    row("MAX (E-box)", "MAX", "chr22 (p<1e-3)", st.loc[M["MAX"].name, "real PWM"], st.loc[M["MAX"].name, "expected"],
        st.loc[M["MAX"].name, "scrambled PWMs (mean)"], "not tested", "CpG context / methylation; partner factors"),
])
summary

### Take-home messages

* **The futility theorem is quantitative.** Comparing the bits a motif carries (R<sub>sequence</sub>) with the bits needed to find its sites in a genome (R<sub>frequency</sub>) predicts the false-positive excess before you scan.
* **Chance hits follow 2 × L × p**, for real motifs and scrambled ones alike. Genome-wide, a real PWM finds barely more hits than a scrambled one. Inside real binding regions, it finds far more.
* **"False" is an operational label.** A site bound in none of many experiments, across cell types and conditions, is very unlikely to be functional, but there are caveats (mappability, untested conditions).
* **What rescues prediction differs per TF.** For GAL4, clusters of sites. For TP53, essentially the motif itself, with binding data across conditions to confirm it. For a short E-box, sequence context (CpG, methylation) and partners. Features such as GC content differ on average but may add little once motif strength is accounted for.
* **Next notebook:** CRM scanning with HMMs (Cluster-Buster and friends) on the *Drosophila* segmentation network, where clusters of *different* TFs define enhancers.

### Extra exercises

1. Add USF1 (JASPAR MA0093.4) and compare it with MAX: a longer E-box matrix with flanks. Does it need fewer bits from context?
2. Change γ in the prediction table to the ~4,300 strongly *and* weakly bound TP53 sites. How does the predicted excess change?
3. Scan yeast with the TP53 matrix (`M["TP53"].set_background(yeast_bg)`). Yeast has no p53. How many hits do you get, and what does that tell you?

### References

* Wasserman WW, Sandelin A (2004) Applied bioinformatics for the identification of regulatory elements. *Nat Rev Genet* 5: 276-287.
* Schneider TD, Stormo GD, Gold L, Ehrenfeucht A (1986) Information content of binding sites on nucleotide sequences. *J Mol Biol* 188: 415-431.
* Aerts S, van Helden J, Sand O, Hassan BA (2007) Fine-tuning enhancer models to predict transcriptional targets across multiple genomes. *PLoS ONE* 2: e1115.
* Verfaillie A, et al. (2016) Multiplex enhancer-reporter assays uncover unsophisticated TP53 enhancer logic. *Genome Res* 26: 882-895.
* Ren B, et al. (2000) Genome-wide location and function of DNA binding proteins. *Science* 290: 2306-2309.
* Gardiner-Garden M, Frommer M (1987) CpG islands in vertebrate genomes. *J Mol Biol* 196: 261-282.
* Oki S, et al. (2018) ChIP-Atlas: a data-mining suite powered by full integration of public ChIP-seq data. *EMBO Rep* 19: e46255.
* JASPAR 2026 (jaspar.elixir.no) for the count matrices.